# GRC model evaluation in Colab: Nanbeige4.2-3B Q&A (stage 1)

This notebook evaluates the model on the team's graded trial questions, **entirely inside your own Colab session**:

1. checks the machine has a GPU;
2. connects your Google Drive and gets the code from GitHub;
3. installs llama.cpp (the pinned build every host uses);
4. gets the model file: converted from Hugging Face the first time, then reused from your Drive;
5. starts the model on this machine only and checks that it works;
6. asks the questions you choose (#1–100) and saves every answer to Drive;
7. scores the answers and shows the HTML report.

**Before the first run** (one time, about 5 min), follow *One-time setup* in
[`colab/README.md`](https://github.com/No1Joycia/grc-model-eval/blob/main/colab/README.md): repo access and a `GH_TOKEN` secret.

**How to run**
1. *Runtime → Change runtime type → **T4 GPU*** (should already be selected).
2. Change the **Settings** form below if you want (question range, thinking on/off, ...).
3. *Runtime → **Run all***. When step 2 asks for Google Drive access, allow it.
4. Keep this tab open. Every log line shows the time and the step, e.g. `[14:02:11 | step 6/7] ...`.

**Time** with the free T4 (estimates, to be confirmed by the first runs): about 15–20 min of setup the first time (mostly the model conversion), 3–5 min in later sessions, plus about 30 s per question.
**Cost:** $0 on the free T4. With Colab Pro it uses compute units the whole time the runtime is connected, so disconnect it when you're done.

**Results** go to Google Drive: `MyDrive/grc-eval/runs/<run name>/` (answers, scores, report, `run_info.json`).
If the session drops, just *Run all* again: finished work is reused and answering continues where it stopped.

In [ ]:
#@title Settings: change these, then Runtime > Run all
#@markdown **Which questions** (#1-33 basic, #34-67 intermediate, #68-100 advanced)
START = 1  #@param {type:"integer"}
END = 100  #@param {type:"integer"}
#@markdown **Model and answer settings**
MODEL = "nanbeige42"  #@param ["nanbeige42"]
QUANT = "q8_0"  #@param ["q8_0", "q4_k_m"]
THINKING = "off"  #@param ["off", "on"]
MAX_TOKENS = 1536  #@param {type:"integer"}
#@markdown **Model source.** Leave both blank for the pinned base model. For a fine-tuned version, give its *merged* Hugging Face repo and revision (and add an `HF_TOKEN` secret if that repo is private).
HF_REPO = ""  #@param {type:"string"}
REVISION = ""  #@param {type:"string"}
#@markdown **Where things go**
DRIVE_FOLDER = "grc-eval"  #@param {type:"string"}
REPO_URL = "https://github.com/No1Joycia/grc-model-eval.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

if not 1 <= START <= END <= 100:
    raise ValueError("START and END must satisfy 1 <= START <= END <= 100")
if HF_REPO and not REVISION:
    raise ValueError("With HF_REPO, also give REVISION (a commit hash or branch name), so the run is reproducible")
print(f"Settings: questions #{START}-#{END} | model {MODEL} ({QUANT}) | thinking {THINKING} | max_tokens {MAX_TOKENS}")
print(f"          source: {HF_REPO + ' @ ' + REVISION if HF_REPO else 'pinned base model (serve/models.json)'}")
print(f"          results: Google Drive > MyDrive/{DRIVE_FOLDER}/runs/")
if THINKING == "on" and MAX_TOKENS < 4096:
    print("NOTE: with thinking on, the model's reasoning also counts against MAX_TOKENS. Use 4096 or more, "
          "or many answers will be cut off (finish_reason=length).")

## Step 1/7: Check the machine

Makes sure this runtime has an NVIDIA GPU before anything is downloaded, and prints what Colab gave you
(GPU, OS, memory, disk). **A T4 is a GPU**, not a TPU; llama.cpp runs on NVIDIA GPUs through CUDA.

*If it stops here:* *Runtime → Change runtime type → T4 GPU → Save*, then *Run all* again.

In [ ]:
#@title Step 1/7: Check the machine (GPU, OS, memory, disk)
if "DRIVE_FOLDER" not in globals():
    raise RuntimeError("The Settings cell (the form at the top) hasn't run in this session. Colab forgets everything when the runtime restarts "
                       "(for example after changing the runtime type). Use Runtime > Run all.")
import os, re, shutil, subprocess, sys, time
NOTEBOOK_STARTED = time.time()

class StopHere(Exception):
    """Stops "Run all" with a message saying what went wrong and what to do."""

def say(msg):
    print(f"[{time.strftime('%H:%M:%S')} | step 1/7] {msg}", flush=True)

print("=" * 80 + "\nSTEP 1/7: Check the machine (GPU, OS, memory, disk)\n" + "=" * 80)
if "google.colab" not in sys.modules:
    raise StopHere("This notebook is written for Google Colab (it uses Colab's Drive and Secrets).")
if not shutil.which("nvidia-smi"):
    raise StopHere("\n\nSTEP 1/7 FAILED: this runtime has no GPU.\n"
                   "What to do: Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all.\n"
                   "If Colab says no GPU is available, try again later (free GPUs aren't guaranteed) or use Colab Pro.")
gpu_line = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                           "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
os_name = dict(l.rstrip().split("=", 1) for l in open("/etc/os-release") if "=" in l)["PRETTY_NAME"].strip('"')
ram_gb = int(re.search(r"MemTotal:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1024 ** 2
say(f"GPU:    {gpu_line}  (name, driver, memory)")
say(f"OS:     {os_name}, Python {sys.version.split()[0]}")
say(f"CPU:    {os.cpu_count()} cores | memory {ram_gb:.1f} GB | free disk {shutil.disk_usage('/content').free / 1e9:.0f} GB")
say("DONE: this machine has a GPU.\n  -> next: step 2/7")

## Step 2/7: Connect Google Drive and get the code

- **Google Drive** keeps the converted model file and your results between sessions (Colab's own disk is wiped
  when the session ends). A pop-up asks for access: choose your account and allow it.
- **The code** is cloned from the private GitHub repo with your `GH_TOKEN` secret. The token is never printed
  or saved in the copy.
- Then it works out the run's name and folders from the settings and `serve/models.json`.

*If it stops here:* the `GH_TOKEN` secret is missing, has no notebook access, or can't read the repo (see the README).

In [ ]:
#@title Step 2/7: Connect Google Drive and get the code
if "NOTEBOOK_STARTED" not in globals():
    raise RuntimeError("Step 1 hasn't run in this session. Colab forgets everything when the runtime restarts "
                       "(for example after changing the runtime type). Use Runtime > Run all.")
import importlib, json
T2 = time.time()
print("=" * 80 + "\nSTEP 2/7: Connect Google Drive and get the code\n" + "=" * 80)

def say(msg):
    print(f"[{time.strftime('%H:%M:%S')} | step 2/7] {msg}", flush=True)

from google.colab import drive, userdata
say("connecting Google Drive (allow access in the pop-up)...")
drive.mount("/content/drive")
DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(DRIVE, exist_ok=True)
say(f"Drive connected: {DRIVE}")

try:
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception as exc:              # SecretNotFoundError, NotebookAccessError
    raise StopHere(f"\n\nSTEP 2/7 FAILED: can't read the GH_TOKEN secret ({type(exc).__name__}).\n"
                   "What to do: click the key icon in Colab's left sidebar, add a secret named GH_TOKEN with your "
                   "GitHub token as the value, switch on 'Notebook access', then run this cell again "
                   "(colab/README.md, One-time setup).") from None
REPO = "/content/grc-model-eval"
authed_url = REPO_URL.replace("https://", f"https://x-access-token:{GH_TOKEN}@")

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(GH_TOKEN, "***").strip()
    if r.returncode:
        raise StopHere(f"\n\nSTEP 2/7 FAILED: git {args[0] if args[0] != '-C' else args[2]} failed:\n{out}\n"
                       "What to do: check that your token can read the repo and hasn't expired, and that "
                       "REPO_URL and BRANCH in the settings are right.")
    return out

if os.path.isdir(f"{REPO}/.git"):
    say("updating the code that is already here...")
    git("-C", REPO, "pull", "--quiet", "--ff-only", authed_url, BRANCH)
else:
    say(f"cloning {REPO_URL} (branch {BRANCH})...")
    git("clone", "--quiet", "--branch", BRANCH, authed_url, REPO)
    git("-C", REPO, "remote", "set-url", "origin", REPO_URL)      # don't keep the token in .git/config
CODE_VERSION = git("-C", REPO, "log", "-1", "--format=%h %ad %s", "--date=short")
say(f"code version: {CODE_VERSION}")
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/colab")
import helpers as h
importlib.reload(h)
h.NOTEBOOK_STARTED = NOTEBOOK_STARTED

try:                                   # only needed for a private fine-tuned model repo
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    say("HF_TOKEN secret found: private Hugging Face repos can be downloaded")
except Exception:
    pass

cfg = json.load(open("serve/models.json"))
LLAMA_BUILD = cfg["llama_cpp"]["build"]
pinned = cfg["models"][MODEL]["source"]
SRC_REPO = HF_REPO or pinned["hf_repo"]
SRC_REV = REVISION or pinned["revision"]
label = MODEL if not HF_REPO else HF_REPO.split("/")[-1]
RUN_NAME = f"{label}-{SRC_REV[:7]}-{QUANT}-think_{THINKING}-{LLAMA_BUILD}"
if MAX_TOKENS != 1536:
    RUN_NAME += f"-mt{MAX_TOKENS}"
RUN_DIR = f"{DRIVE}/runs/{RUN_NAME}"
MODEL_KEY = f"{SRC_REPO.replace('/', '__')}-{SRC_REV[:7]}-{LLAMA_BUILD}"
DRIVE_MODEL_DIR = f"{DRIVE}/models/{MODEL_KEY}"      # kept between sessions
LOCAL_MODEL_DIR = f"/content/models/{MODEL_KEY}"     # fast local disk, used while serving
os.makedirs(RUN_DIR, exist_ok=True)
say(f"run name:    {RUN_NAME}")
say(f"run folder:  {RUN_DIR}")
say(f"model cache: {DRIVE_MODEL_DIR}")
h.TIMINGS["2. Connect Google Drive and get the code"] = round(time.time() - T2)
say("DONE: Drive connected and code ready.\n  -> next: step 3/7")

## Step 3/7: Install llama.cpp

Downloads llama.cpp's own **prebuilt Linux CUDA release** of the pinned build (about 740 MB, 1–2 min), the same build
every host uses (`serve/models.json`), and checks that it can see the GPU.

*Fallback:* if the prebuilt version can't use this GPU (Colab's NVIDIA driver too old for CUDA 12.8), the cell compiles
the same build here instead (about 20–40 min, first time only) and keeps a copy in your Drive. This path hasn't been
tested in Colab yet: if it fails, send the output to Joy.

In [ ]:
#@title Step 3/7: Install llama.cpp (pinned build, NVIDIA CUDA)
if "RUN_DIR" not in globals():
    raise RuntimeError("Step 2 hasn't run in this session. Colab forgets everything when the runtime restarts "
                       "(for example after changing the runtime type). Use Runtime > Run all.")
LLAMA_DIR = f"/content/llama.cpp/{LLAMA_BUILD}"
h.step(3, f"Install llama.cpp {LLAMA_BUILD} (NVIDIA CUDA build)", f"""
Downloads llama.cpp's prebuilt Linux CUDA release into {LLAMA_DIR} (skipped if it's already there),
then checks that llama.cpp can see the GPU.""")
os.environ["LLAMA_CPP_DIR"] = LLAMA_DIR
os.environ["LD_LIBRARY_PATH"] = f"{LLAMA_DIR}:{os.environ.get('LD_LIBRARY_PATH', '')}"
GPU = h.gpu_info()
h.log(f"GPU {GPU['name']}, driver {GPU['driver']} (supports CUDA up to {GPU['driver_cuda']}); "
      "the prebuilt llama.cpp is built with CUDA 12.8")
code, _ = h.run(["bash", "serve/install_llamacpp.sh"], env={"VARIANT": "cuda"}, check=False)
if not os.path.exists(f"{LLAMA_DIR}/llama-server"):
    h.fail(f"the llama.cpp download failed (exit code {code})",
           "read the output above (usually a network problem), then run this cell again")
DEVICES, out = h.cuda_devices(LLAMA_DIR)
if not DEVICES:
    h.log("the prebuilt llama.cpp can't use this GPU. What it said:")
    for line in out[-8:]:
        print("    " + line)
    h.log("FALLBACK: compiling llama.cpp for this GPU instead")
    h.build_llamacpp_from_source(LLAMA_BUILD, LLAMA_DIR, f"{DRIVE}/cache", GPU["compute_cap"])
    DEVICES, out = h.cuda_devices(LLAMA_DIR)
    if not DEVICES:
        h.fail("llama.cpp still can't see the GPU", "send this cell's output to Joy",
               "\n".join("    " + line for line in out[-10:]))
_, version = h.run([f"{LLAMA_DIR}/llama-server", "--version"])
LLAMA_VERSION = next((l for l in version if "build" in l), version[0]).strip()
if f"build {LLAMA_BUILD[1:]}," not in LLAMA_VERSION:
    h.fail(f"llama.cpp reports '{LLAMA_VERSION}', not build {LLAMA_BUILD}",
           f"delete {LLAMA_DIR} (or restart the runtime) and run this cell again")
h.done(f"{LLAMA_VERSION}; it sees {DEVICES[0]}")

## Step 4/7: Get the model file

The model runs from a **GGUF file**, llama.cpp's model format.

- **First session:** downloads the checkpoint from Hugging Face (8.4 GB), converts it to BF16 and then to the chosen
  quantization (`serve/get_gguf.sh` → `convert/to_gguf.sh`), and saves only that file to your Drive.
  About 10–15 min.
- **Later sessions:** copies the file from your Drive to this machine's disk and checks it: 1–3 min.

Every file is checked against the SHA-256 in `serve/models.json`, so every teammate evaluates exactly the same file.
The first Colab conversion has no Linux reference yet: the cell then prints the value for Joy to add.

In [ ]:
#@title Step 4/7: Get the model file (convert once, then reuse your Drive copy)
if "LLAMA_VERSION" not in globals():
    raise RuntimeError("Step 3 hasn't run in this session. Colab forgets everything when the runtime restarts "
                       "(for example after changing the runtime type). Use Runtime > Run all.")
h.step(4, "Get the model file", f"""
{QUANT} of {SRC_REPO} @ {SRC_REV[:7]}
Drive copy:  {DRIVE_MODEL_DIR}
Local copy:  {LOCAL_MODEL_DIR} (llama.cpp reads the model from local disk, not from Drive)""")
gguf_name = cfg["models"][MODEL]["gguf"][QUANT]
drive_file, local_file = f"{DRIVE_MODEL_DIR}/{gguf_name}", f"{LOCAL_MODEL_DIR}/{gguf_name}"
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)
if os.path.exists(drive_file) and not os.path.exists(local_file):
    h.log(f"copying {gguf_name} ({os.path.getsize(drive_file) / 1e9:.1f} GB) from Drive to local disk...")
    t = time.time()
    shutil.copy(drive_file, local_file + ".partial")
    os.replace(local_file + ".partial", local_file)
    h.log(f"copied in {h.fmt(time.time() - t)}")
elif not os.path.exists(drive_file):
    h.log("no copy in your Drive yet: converting from Hugging Face (first time only)")
if not shutil.which("uv"):
    h.log("installing uv (the Python package installer the converter uses)")
    h.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
CHATTY = r"^(INFO:hf-to-gguf:|INFO:gguf|\[\s*\d+/\s*\d+\]|llama_model_loader: |llama_model_quantize|main: |load_imatrix)"
_, lines = h.run(["bash", "serve/get_gguf.sh", MODEL, QUANT, LOCAL_MODEL_DIR, SRC_REPO, SRC_REV],
                 env={"SCRATCH": "/content/scratch"}, quiet=CHATTY,
                 on_fail="read the [get_gguf] lines above: they say what failed and what to do")
result = h.parse_result(lines)
GGUF, GGUF_SHA, GGUF_CHECK = result["file"], result["sha256"], result["check"]
if not os.path.exists(drive_file):
    h.log(f"saving the model file to your Drive for next time ({os.path.getsize(GGUF) / 1e9:.1f} GB)...")
    try:
        os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
        shutil.copy(GGUF, drive_file + ".partial")
        os.replace(drive_file + ".partial", drive_file)
        h.log("saved")
    except OSError as exc:
        h.log(f"WARNING: couldn't save it to Drive ({exc}). This run continues; the next session converts again.")
if GGUF_CHECK == "new-reference":
    print("\n" + "!" * 80)
    print("FIRST CONVERSION ON THIS KIND OF MACHINE. Please send this line to Joy, so it becomes the team")
    print(f"reference in serve/models.json:   sha256.{QUANT}.\"Linux-x86_64\": \"{GGUF_SHA}\"")
    print("!" * 80 + "\n")
h.done(f"{gguf_name} ({os.path.getsize(GGUF) / 1e9:.1f} GB), sha256 {GGUF_SHA[:16]}..., check: {GGUF_CHECK}")

## Step 5/7: Start the model and check it works

Starts **llama-server** in the background on `127.0.0.1` at a free local port (8181 unless it's taken; Colab itself
uses 8080): on this machine only, nothing is exposed to the
internet. It uses the shared settings in `serve/models.json` (64K context, flash attention, all layers on the GPU,
Nanbeige's chat template). While it loads, the cell prints what the server is doing every 15 s.

Then `serve/smoke_test.py` checks four things: health, model list, one chat answer (with thinking, so it can take a
minute) and one tool call. Q&A doesn't use tools, so a failed tool-call check only gives a warning.

In [ ]:
#@title Step 5/7: Start the model server and run the smoke test
if "GGUF" not in globals():
    raise RuntimeError("Step 4 hasn't run in this session. Colab forgets everything when the runtime restarts "
                       "(for example after changing the runtime type). Use Runtime > Run all.")
h.step(5, "Start the model server and run the smoke test", """
Starts llama-server on 127.0.0.1 at a free port (this machine only) with the settings in serve/models.json,
waits until the model is loaded, then checks: health, models, one chat answer, one tool call.""")
h.kill_stray_servers()
if "SERVER" in globals():
    h.stop_server(SERVER)
SERVER_LOG = "/content/llama-server.log"
PORT = h.free_port(8181)                      # Colab's own runtime listens on 8080
BASE_URL = f"http://127.0.0.1:{PORT}/v1"
SERVER = h.start_server(REPO, MODEL, QUANT, GGUF, SERVER_LOG, port=PORT)
h.log(f"llama-server started on port {PORT} (process {SERVER.pid}); its log: {SERVER_LOG}")
h.wait_ready(SERVER, SERVER_LOG, port=PORT)
h.log("what the server reports about the GPU:")
for line in open(SERVER_LOG, errors="replace"):
    if re.search(r"offloaded \d+/\d+ layers|CUDA0 model buffer size|KV buffer size|n_ctx\s+=", line):
        print("    " + line.strip())
h.log("running the smoke test (serve/smoke_test.py)...")
_, lines = h.run([sys.executable, "serve/smoke_test.py", "--base-url", BASE_URL, "--model", MODEL], check=False)
failed = [l for l in lines if l.startswith("FAIL")]
if any(not l.startswith("FAIL  tool call") for l in failed):
    h.fail("the smoke test failed", "read the FAIL lines above and the server log; then rerun this step",
           "\n".join("    " + l for l in h.tail(SERVER_LOG)))
if failed:
    h.log("WARNING: the tool-call check failed. Q&A questions don't use tools, so the evaluation goes on; tell Joy.")
h.done("the model is loaded on the GPU and answers")

## Step 6/7: Answer the questions

`trial_eval/run_trial.py` sends each question in your range to the local model, with the team's fixed system prompt,
temperature 0 and your thinking setting, and **saves each answer to Drive as soon as it arrives**.

- The first line is a time **ESTIMATE**; before any answer is timed it's a rough guess, and it improves on later runs.
- Then `asking #N ...` before each question and one line after it: tokens, seconds, speed, `stop`
  (finished) or `length` (cut off by `MAX_TOKENS`), and the time left.
- The last line is **DONE** with the total time.

If the session drops, *Run all* again: questions already answered are skipped.

In [ ]:
#@title Step 6/7: Answer the questions (saved to Drive one by one)
if "SERVER" not in globals():
    raise RuntimeError("Step 5 hasn't run in this session. Colab forgets everything when the runtime restarts "
                       "(for example after changing the runtime type). Use Runtime > Run all.")
ANSWERS = f"{RUN_DIR}/answers.jsonl"
h.step(6, f"Answer questions #{START}-#{END}", f"""
Asks the local model each question (trial_eval/run_trial.py) and saves every answer at once to
{ANSWERS}""")
h.run([sys.executable, "trial_eval/run_trial.py", "--base-url", BASE_URL,
       "--model", MODEL, "--thinking", THINKING, "--max-tokens", str(MAX_TOKENS),
       "--start", str(START), "--end", str(END), "--out", ANSWERS],
      on_fail="answers so far are saved. If the server stopped, rerun step 5, then this step")
n_answers = sum(1 for line in open(ANSWERS) if line.strip())
h.done(f"{n_answers} answers in {ANSWERS}")

## Step 7/7: Score the answers and show the report

Stops the model (frees the GPU), then `trial_eval/score_trial.py` compares each answer in your range with the graded
reference answers. The first time in a session it downloads the bge-m3 embedding model (2.2 GB, about 1 min).
It writes the scores (CSV), a summary (JSON) and the **HTML report** into the run folder, plus `run_info.json`
(GPU, llama.cpp build, model file hash, settings, timings). The report is shown below and downloaded.

**How to read the scores:** `sem_norm` is the main number: 0 ≈ as similar as an answer to a different question,
1 ≈ as similar as another passing teacher answer. It measures similarity, not correctness, so read the
lowest-scoring answers yourself. More in `trial_eval/README.txt`.

In [ ]:
#@title Step 7/7: Score the answers and make the report
if "ANSWERS" not in globals():
    raise RuntimeError("Step 6 hasn't run in this session. Colab forgets everything when the runtime restarts "
                       "(for example after changing the runtime type). Use Runtime > Run all.")
h.step(7, "Score the answers and make the report", """
Stops the model server, scores the answers (trial_eval/score_trial.py), writes run_info.json,
then shows and downloads the HTML report.""")
h.stop_server(SERVER)
shutil.copy(SERVER_LOG, f"{RUN_DIR}/llama-server.log")
for module, package in (("rouge_score", "rouge-score"), ("sentence_transformers", "sentence-transformers")):
    try:
        importlib.import_module(module)
    except ImportError:
        h.log(f"installing {package}...")
        h.run([sys.executable, "-m", "pip", "install", "--quiet", package])
h.log("scoring (the first time in a session this downloads bge-m3, 2.2 GB)...")
_, lines = h.run([sys.executable, "trial_eval/score_trial.py", "--answers", ANSWERS,
                  "--start", str(START), "--end", str(END)],
                 quiet=r"^(Batches|Loading weights|Warning: You are sending unauthenticated)")
outputs = {l.split(" -> ")[0].strip(): l.split(" -> ", 1)[1].strip() for l in lines if " -> " in l}
REPORT, SUMMARY = outputs["HTML report"], outputs["summary"]
summary = json.load(open(SUMMARY))
h.TIMINGS[f"7. {h._step['title']}"] = round(time.time() - h._step["started"])
run_info = {
    "run": RUN_NAME,
    "finished": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "questions": {"start": START, "end": END, "answers_in_file": sum(1 for l in open(ANSWERS) if l.strip())},
    "model": {"name": MODEL, "hf_repo": SRC_REPO, "revision": SRC_REV, "quant": QUANT,
              "gguf": os.path.basename(GGUF), "sha256": GGUF_SHA, "sha256_check": GGUF_CHECK},
    "settings": {"thinking": THINKING, "max_tokens": MAX_TOKENS, "temperature": 0},
    "llama_cpp": LLAMA_VERSION,
    "gpu": {"device": DEVICES[0], **GPU},
    "system": h.system_info(),
    "code_version": CODE_VERSION,
    "step_seconds": h.TIMINGS,
    "scores_overall": summary["overall"],
}
h.write_json(f"{RUN_DIR}/run_info.json", run_info)
o = summary["overall"]
h.done(f"{o['n']} answers scored: sem_norm {o['sem_norm']}, sem_best {o['sem_best']}, "
       f"rougeL {o['rougeL']}, cut off {o['truncated']}")
print(f"\nEverything for this run is in Google Drive: {RUN_DIR}")
for name in sorted(os.listdir(RUN_DIR)):
    print(f"    {name}")
print(f"\nTotal time: {h.fmt(time.time() - NOTEBOOK_STARTED)}. Report below (also downloaded).")
from IPython.display import HTML, display
from google.colab import files
display(HTML(open(REPORT).read()))
files.download(REPORT)

## When you're done

- **Free the machine:** *Runtime → Disconnect and delete runtime*. On Colab Pro this stops compute units being used.
- **Share the results:** copy the run folder (`MyDrive/grc-eval/runs/<run name>/`) into the team's shared Drive folder.
  Its `run_info.json` says exactly what was evaluated.
- **Another range** (e.g. #34–67): change `START` / `END` and *Run all*. Answers go into the same run folder; the report
  covers the range you set.
- **Different settings** (thinking, quantization, max tokens, another model): they get their own run folder automatically.